## Bài 1: Masked Language Modeling

In [ ]:
from transformers import pipeline

mask_filler = pipeline("fill-mask")
input_sentence = "Hanoi is the [MASK] of Vietnam."
predictions = mask_filler(input_sentence, top_k=5)

print(f"Câu gốc: {input_sentence}")
for pred in predictions:
    print(f"Dự đoán: '{pred['token_str']}' với độ tin cậy: {pred['score']:.4f}")
    print(f" -> Câu hoàn chỉnh: {pred['sequence']}")

### Trả lời câu hỏi
1. Mô hình dự đoán đúng từ **capital** (thường đúng trong top-1).
2. BERT là mô hình **Encoder-only**, sử dụng **bidirectional attention**, phù hợp với việc dự đoán token bị che.

## Bài 2: Text Generation với GPT

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation")
prompt = "The best thing about learning NLP is"
generated_texts = generator(prompt, max_length=50, num_return_sequences=1)

print(f"Câu mồi: '{prompt}'")
for text in generated_texts:
    print("Văn bản được sinh ra:")
    print(text['generated_text'])

### Trả lời câu hỏi
1. Kết quả sinh ra hợp lý tùy vào GPT-2.
2. GPT dùng **causal attention**, được huấn luyện cho **next token prediction**, phù hợp sinh văn bản.

## Bài 3: Sentence Embedding với Mean Pooling

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

sentences = ["This is a sample sentence."]
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
attention_mask = inputs['attention_mask']

mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
sentence_embedding = sum_embeddings / sum_mask

print("Vector biểu diễn của câu:")
print(sentence_embedding)
print("Kích thước:", sentence_embedding.shape)

### Trả lời câu hỏi
1. Vector có kích thước **768**, bằng với **hidden_size** của BERT base.
2. **attention_mask** loại bỏ padding để tính mean chính xác.